<a href="https://colab.research.google.com/github/Avichatt/-Personal-Finance-Calculator/blob/main/Custom_Tokenizer_using_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -y tokenizer transformers
!pip install datasets torch



Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -y


In [2]:
from datasets import load_dataset
dataset = load_dataset("imdb", split="train[:2000]")
corpus = dataset["text"]
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})


In [3]:
print(corpus)

Column(['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far bet

In [11]:
from transformers import BertTokenizerFast

# start from an existing tokenizer shell
# trained on IMDB dataset
# tokenizer contains = corpus, config, vocabulary

base_tokenizer= BertTokenizerFast.from_pretrained("bert-base-uncased")
tokenizer = base_tokenizer.train_new_from_iterator(corpus, 52000)
tokenizer.save_pretrained("tokenizer")



In [12]:
# Open the file in read mode
with open("tokenizer/vocab.txt", "r") as f:
    vocab_content = f.read()   # read the entire file into a string

# Print the content
print(vocab_content)


superpowers
announcer
duplicating
##acial
glittering
##estown
briskly
sensuous
##cies
infection
participating
swedes
##ity
##iere
henner
spite
extinct
pigs
sawto
bib
ideal
##thall
depardieu
coilition
loyalist
##raina
muriel
calamity
##ersive
signifying
bouncing
dislikably
courage
eff
moe
froze
panacea
##warts
inherited
hatter
hearts
coupling
interaction
rooney
##oise
##ivity
batch
kiya
fiascos
dynasty
dialouge
##olith
giallio
disor
##blers
partner
tina
##eting
torpedo
turns
brewsters
tot
bonnevie
##bys
aficion
##regious
none
relatable
oosh
healthy
hawking
manson
##tled
duration
bumper
negotiation
uterus
acoly
anore
cheek
dehavilland
chased
ass
animates
chattering
unopposed
affleck
notebook
##rand
##ails
emerging
tamp
celebrating
plasticine
alpha
frights
matata
##oix
##you
dogsbody
hanson
innocent
ridicul
oakley
philosoph
##bun
pochath
policies
wool
tooth
jewish
##ged
anneliza
thom
killers
##inative
camcorder
##itarian
undercuts
wont
chilkats
renov
washing
eschewed
intrude
recru
##arms


In [13]:
from transformers import BertConfig, BertForMaskedLM

# Example: assume you already have a tokenizer
# tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

config = BertConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_size=128,
    num_hidden_layers=2,
    num_attention_heads=4,
    intermediate_size=512,
    max_position_embeddings=256
)

model = BertForMaskedLM(config)  # pass the config object
model.eval()


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32848, 128, padding_idx=0)
      (position_embeddings): Embedding(256, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-1): 2 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=128, out_features=128, bias=True)
              (key): Linear(in_features=128, out_features=128, bias=True)
              (value): Linear(in_features=128, out_features=128, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=128, out_features=128, bias=True)
              (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_aff

In [17]:
import torch

def min_perplexity(model, tokenizer, texts, max_length=128):
    model.eval()
    losses = []

    for text in texts:
        # Encode text
        enc = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        )

        # Labels: copy input ids
        labels = enc["input_ids"].clone()

        # Create probability matrix for masking (15% chance)
        probability_matrix = torch.full(labels.shape, 0.15)

        # Randomly select positions to mask
        masked_indices = torch.bernoulli(probability_matrix).bool()

        # Ignore unmasked positions in loss
        labels[~masked_indices] = -100

        with torch.no_grad():
            outputs = model(**enc, labels=labels)
            loss = outputs.loss
            losses.append(loss.item())

    # Perplexity = exp(mean loss)

    return torch.exp(torch.tensor(losses).mean()).item()



In [18]:
# Compute perplexity for the corpus
ppl = min_perplexity(model, tokenizer, corpus)

print("Perplexity:", ppl)



Perplexity: nan
